# Advanced Model Risk Management
## Notebook for Tutorials 1–2 — The Phoenix Autocall

This notebook accompanies the tutorial. Each section serves a block of questions of the subject.

**Guiding principle.** All the models used here are calibrated to *the same* market data. Any
price divergence you observe is therefore neither an implementation error nor a parameter
error: it is model risk in the strict sense.

**Your term sheet.** Set `GROUP` and `TEAM` in the parameter cell of §0, then run the whole
notebook: every number then refers to the barriers of your team's term sheet.

**Who answers.** ● team (team answer sheet, including its Decision tab) — ◆ individual,
mandatory (individual written assignment) — ○ optional (not graded).

**Precision.** Monte Carlo prices and break-even coupons are printed with the number of paths
and a standard error (SE). Report both in your answer sheet.

**Total run time**: about 3 minutes (almost two of them for §4ter and §5).

---
### Contents
| § | Content | Questions |
|---|---|---|
| 0 | Toolbox and parameters (your term sheet) | — |
| 1 | The reference market: implied volatility surface | Q10 ● |
| 2 | Dupire local volatility and Gyöngy's theorem | Q10 ● |
| 3 | The European-barrier put: the models agree | Q11 ● |
| 4 | The full Phoenix: the models diverge | Q12–Q14 ●, Q15 ◆ |
| 4bis | Measuring the forward skew | Q16 ● |
| 4ter | M4: the local-stochastic model | Q17 ○, Q18 ● |
| 5 | Sign of vega | Q19–Q21 ◆ |
| 6 | Choosing a model by its greeks: hedging backtest | Q22–Q24 ● |
| 7 | Delta at the barriers and *barrier shift* (run by your team) | Q25–Q26 ●, Q27 ○ |
| 8 | Block D: a pricer written by an AI, to validate | D1–D5 ● |

**AI tools are not allowed** to answer the questions (team or individual work), except to help write the Block D tests (§8).


## §0 — Toolbox

The next cells contain the whole numerical engine: Heston characteristic function, COS method,
Dupire's formula, Monte Carlo simulators, Crank–Nicolson PDE solver and the Phoenix payoff.

You do not need to read them in detail to do the tutorial. They are provided so that everything
can be checked and modified.

**Recent numpy.** `np.trapz` was renamed `np.trapezoid` in numpy 2.0 and has since been removed.
The first cell defines `_trapz`, which uses whichever of the two exists, so the notebook runs
with old and new versions of numpy alike.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq
from scipy.linalg import solve_banded
from scipy.interpolate import RegularGridInterpolator

# numpy >= 2.0 provides np.trapezoid (np.trapz is removed in recent versions); older numpy only np.trapz
_trapz = getattr(np, 'trapezoid', None) or np.trapz

np.set_printoptions(suppress=True, precision=4)
plt.rcParams.update({'figure.figsize': (10, 4.2), 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 10})

In [ ]:
# ---------- Heston: characteristic function and density by the COS method ----------

def heston_cf(u, T, r, q, v0, kappa, theta, sigma, rho):
    u = np.asarray(u, dtype=complex); iu = 1j * u
    d = np.sqrt((rho*sigma*iu - kappa)**2 + sigma**2 * (iu + u**2))
    g = (kappa - rho*sigma*iu - d) / (kappa - rho*sigma*iu + d)
    e = np.exp(-d*T)
    C = (kappa*theta/sigma**2) * ((kappa - rho*sigma*iu - d)*T
                                  - 2*np.log((1 - g*e)/(1 - g)))
    D = ((kappa - rho*sigma*iu - d)/sigma**2) * ((1 - e)/(1 - g*e))
    return np.exp(iu*(r - q)*T + C + D*v0)

def _cos_range(T, r, q, v0, theta, L=12.0):
    vbar = max(theta, v0); s = np.sqrt(vbar*T + 1e-12)
    m = (r - q - 0.5*vbar)*T
    return m - L*s, m + L*s

def heston_density(x, T, r, q, v0, kappa, theta, sigma, rho, N=2048, L=12.0):
    a, b = _cos_range(T, r, q, v0, theta, L)
    k = np.arange(N); u = k*np.pi/(b - a)
    cf = heston_cf(u, T, r, q, v0, kappa, theta, sigma, rho)
    Fk = (2.0/(b - a)) * np.real(cf*np.exp(-1j*u*a)); Fk[0] *= 0.5
    x = np.atleast_1d(x)
    dens = np.cos(np.outer(x - a, u)) @ Fk
    return np.maximum(np.where((x < a) | (x > b), 0.0, dens), 0.0)

def heston_european(payoff, T, S0, r, q, hp, n_grid=4000, L=12.0):
    """Price of a payoff g(S_T) by integration against the COS density."""
    a, b = _cos_range(T, r, q, hp['v0'], hp['theta'], L)
    x = np.linspace(a, b, n_grid)
    dens = heston_density(x, T, r, q, L=L, **hp)
    return np.exp(-r*T) * _trapz(payoff(S0*np.exp(x))*dens, x)

def heston_call(K, T, S0, r, q, hp, n_grid=4000, L=12.0):
    """Calls for a vector of strikes: a single density per maturity."""
    K = np.atleast_1d(np.asarray(K, float))
    a, b = _cos_range(T, r, q, hp['v0'], hp['theta'], L)
    x = np.linspace(a, b, n_grid)
    dens = heston_density(x, T, r, q, L=L, **hp)
    ST = S0*np.exp(x)
    pay = np.maximum(ST[None, :] - K[:, None], 0.0)
    return np.exp(-r*T) * _trapz(pay*dens[None, :], x, axis=1)

# ---------- Black-Scholes and implied volatility ----------

def bs_call(S0, K, T, r, q, vol):
    if T <= 0 or vol <= 0:
        return max(S0*np.exp(-q*T) - K*np.exp(-r*T), 0.0)
    d1 = (np.log(S0/K) + (r - q + 0.5*vol**2)*T)/(vol*np.sqrt(T))
    d2 = d1 - vol*np.sqrt(T)
    return S0*np.exp(-q*T)*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def implied_vol(price, S0, K, T, r, q):
    intr = max(S0*np.exp(-q*T) - K*np.exp(-r*T), 0.0)
    if price <= intr + 1e-10: return np.nan
    try:    return brentq(lambda v: bs_call(S0, K, T, r, q, v) - price, 1e-4, 5.0, xtol=1e-10)
    except ValueError: return np.nan

In [ ]:
# ---------- Local volatility surface (Dupire's formula in total implied variance) ----------

class LocalVolSurface:
    def __init__(self, ys, Ts, w, floor=0.01, cap=1.5):
        self.ys, self.Ts, self.w = ys, Ts, w
        dw_dT   = np.gradient(w, Ts, axis=0)
        dw_dy   = np.gradient(w, ys, axis=1)
        d2w_dy2 = np.gradient(dw_dy, ys, axis=1)
        Y = ys[None, :]*np.ones((len(Ts), 1)); W = np.maximum(w, 1e-8)
        denom = (1 - (Y/W)*dw_dy
                 + 0.25*(-0.25 - 1/W + Y**2/W**2)*dw_dy**2
                 + 0.5*d2w_dy2)
        self.locvar = np.clip(dw_dT/np.maximum(denom, 1e-6), floor**2, cap**2)

    def __call__(self, S, t, F):
        y = np.log(np.maximum(S, 1e-8)/F)
        it = int(np.clip(np.searchsorted(self.Ts, t), 0, len(self.Ts) - 1))
        return np.sqrt(np.interp(y, self.ys, self.locvar[it]))

def build_local_vol(S0, r, q, hp, Ts, ys, iv_bump=0.0):
    """Builds the local volatility surface from the reference model.
    iv_bump: parallel shift of the implied volatility (used to compute vega)."""
    w = np.zeros((len(Ts), len(ys)))
    for i, T in enumerate(Ts):
        F = S0*np.exp((r - q)*T); Ks = F*np.exp(ys)
        for j, (K, p) in enumerate(zip(Ks, heston_call(Ks, T, S0, r, q, hp))):
            iv = implied_vol(p, S0, K, T, r, q)
            w[i, j] = ((iv + iv_bump)**2)*T if np.isfinite(iv) else np.nan
        bad = ~np.isfinite(w[i])
        if bad.any(): w[i, bad] = np.interp(ys[bad], ys[~bad], w[i, ~bad])
    return LocalVolSurface(ys, Ts, w), w

In [ ]:
# ---------- Monte Carlo simulators ----------
# To save memory, the simulators only return what the payoff needs: the spot on the
# observation dates, and the running minimum between two dates (needed for the variant
# with a protection barrier observed at each close). Pass obs_idx=None to get the full
# paths (used in §6 for the hedging backtest).
# Antithetic variates: path j and path j + n_paths/2 use opposite Gaussian draws. The
# standard errors printed below are therefore computed on pair averages (see mc_se).

def _pack(S, i, obs_idx, k, ov, rm, run):
    """Stores the value and the running minimum if step i+1 is an observation date."""
    run = np.minimum(run, S)
    if k < len(obs_idx) and (i + 1) == obs_idx[k]:
        ov[k] = S; rm[k] = run; run = np.full_like(S, np.inf); k += 1
    return k, run

def simulate_heston(S0, r, q, hp, T, n_steps, n_paths, seed=0,
                    obs_idx=None, return_var=False):
    rng = np.random.default_rng(seed); dt = T/n_steps; m = n_paths//2
    v0, ka, th, si, rh = hp['v0'], hp['kappa'], hp['theta'], hp['sigma'], hp['rho']
    S = np.full(n_paths, float(S0)); v = np.full(n_paths, float(v0))
    full = obs_idx is None
    if full:
        out = np.empty((n_steps+1, n_paths)); var = np.empty((n_steps+1, n_paths))
        out[0], var[0] = S, v
    else:
        ov = np.empty((len(obs_idx), n_paths)); rm = np.empty_like(ov)
        k = 0; run = np.full(n_paths, np.inf)
    for i in range(n_steps):
        z1 = rng.standard_normal(m); z2 = rng.standard_normal(m)
        z1 = np.concatenate([z1, -z1]); z2 = np.concatenate([z2, -z2])
        dW1 = z1*np.sqrt(dt); dW2 = (rh*z1 + np.sqrt(1 - rh**2)*z2)*np.sqrt(dt)
        vp = np.maximum(v, 0.0)
        S = S*np.exp((r - q - 0.5*vp)*dt + np.sqrt(vp)*dW1)
        v = v + ka*(th - vp)*dt + si*np.sqrt(vp)*dW2
        if full: out[i+1], var[i+1] = S, np.maximum(v, 0.0)
        else:    k, run = _pack(S, i, obs_idx, k, ov, rm, run)
    if full: return (out, var) if return_var else out
    return ov, rm

def simulate_localvol(S0, r, q, lv, T, n_steps, n_paths, seed=0, obs_idx=None):
    rng = np.random.default_rng(seed); dt = T/n_steps; m = n_paths//2
    S = np.full(n_paths, float(S0)); full = obs_idx is None
    if full:
        out = np.empty((n_steps+1, n_paths)); out[0] = S
    else:
        ov = np.empty((len(obs_idx), n_paths)); rm = np.empty_like(ov)
        k = 0; run = np.full(n_paths, np.inf)
    for i in range(n_steps):
        t = i*dt; sig = lv(S, t, S0*np.exp((r - q)*t))
        z = rng.standard_normal(m); z = np.concatenate([z, -z])
        S = S*np.exp((r - q - 0.5*sig**2)*dt + sig*np.sqrt(dt)*z)
        if full: out[i+1] = S
        else:    k, run = _pack(S, i, obs_idx, k, ov, rm, run)
    return out if full else (ov, rm)

def simulate_bs(S0, r, q, vol, T, n_steps, n_paths, seed=0, obs_idx=None):
    rng = np.random.default_rng(seed); dt = T/n_steps; m = n_paths//2
    S = np.full(n_paths, float(S0)); full = obs_idx is None
    if full:
        out = np.empty((n_steps+1, n_paths)); out[0] = S
    else:
        ov = np.empty((len(obs_idx), n_paths)); rm = np.empty_like(ov)
        k = 0; run = np.full(n_paths, np.inf)
    for i in range(n_steps):
        z = rng.standard_normal(m); z = np.concatenate([z, -z])
        S = S*np.exp((r - q - 0.5*vol**2)*dt + vol*np.sqrt(dt)*z)
        if full: out[i+1] = S
        else:    k, run = _pack(S, i, obs_idx, k, ov, rm, run)
    return out if full else (ov, rm)

In [ ]:
# ---------- Phoenix payoff (Monte Carlo) ----------

def phoenix_mc(sim, S0, r, obs, Ba, Bc, coupon, Bp, american=False, memory=False):
    """sim = (values on the observation dates, running minima between dates).
    american=True: protection barrier observed at each simulated close (running minimum).
    Returns (price, annuity of the coupon strip, redemption leg)."""
    ov, rm = sim
    n = ov.shape[1]
    alive = np.ones(n, bool); pv_c = np.zeros(n); pv_r = np.zeros(n)
    missed = np.zeros(n); knocked = np.zeros(n, bool)
    for k, d in enumerate(obs):
        perf = ov[k]/S0; df = np.exp(-r*d)
        if american: knocked |= (rm[k]/S0 < Bp)
        pay = alive & (perf >= Bc)
        if memory:
            pv_c += np.where(pay, coupon*(1 + missed)*df, 0.0)
            missed = np.where(pay, 0.0, missed + alive.astype(float))
        else:
            pv_c += np.where(pay, coupon*df, 0.0)
        if k < len(obs) - 1:
            called = alive & (perf >= Ba)
            pv_r += np.where(called, df, 0.0); alive &= ~called
        else:
            ki = knocked if american else (perf < Bp)
            pv_r += np.where(alive, np.where(ki, np.minimum(perf, 1.0), 1.0)*df, 0.0)
    return (pv_c + pv_r).mean(), pv_c.mean()/coupon, pv_r.mean()

def break_even_coupon(annuity, redemption_leg, par=1.0):
    return (par - redemption_leg)/annuity

# ---------- Monte Carlo standard errors ----------

def phoenix_paths(sim, S0, r, obs, Ba, Bc, Bp, american=False):
    """Same payoff as phoenix_mc (no memory), but path by path and per unit of coupon.
    Returns (a, red): a[i] = sum over dates of (coupon indicator x discount factor) on path i,
    red[i] = discounted redemption leg of path i. Their means are the annuity and the
    redemption leg returned by phoenix_mc."""
    ov, rm = sim
    n = ov.shape[1]
    alive = np.ones(n, bool); a = np.zeros(n); red = np.zeros(n); knocked = np.zeros(n, bool)
    for k, d in enumerate(obs):
        perf = ov[k]/S0; df = np.exp(-r*d)
        if american: knocked |= (rm[k]/S0 < Bp)
        a += np.where(alive & (perf >= Bc), df, 0.0)
        if k < len(obs) - 1:
            called = alive & (perf >= Ba)
            red += np.where(called, df, 0.0); alive &= ~called
        else:
            ki = knocked if american else (perf < Bp)
            red += np.where(alive, np.where(ki, np.minimum(perf, 1.0), 1.0)*df, 0.0)
    return a, red

def mc_se(x):
    """Standard error of the mean of x simulated with antithetic pairs: path j and path
    j + n/2 are averaged first, and the n/2 pair averages are i.i.d."""
    x = np.asarray(x, float); m = len(x)//2
    y = 0.5*(x[:m] + x[m:2*m])
    return y.std(ddof=1)/np.sqrt(m)

def coupon_influence(a, red, par=1.0):
    """Break-even coupon c = (par - mean red)/mean a and its per-path influence (delta method):
    dc ~ -(d mean(red) + c d mean(a))/mean(a), i.e. psi_i = -(red_i + c a_i - par)/mean(a)."""
    c = (par - red.mean())/a.mean()
    return c, -(red + c*a - par)/a.mean()

def break_even_coupon_se(a, red, par=1.0):
    """(c, SE of c), both per half-year: SE(c) = SE of the mean of (red_i + c a_i), / mean(a).
    Multiply both by 2 for the annual coupon."""
    c, psi = coupon_influence(a, red, par)
    return c, mc_se(psi)

In [ ]:
# ---------- PDE solvers (Crank-Nicolson in log-spot) ----------

def _cn_step(V, S, x, dt, r, q, sig):
    s2 = sig**2; dx = x[1] - x[0]
    A = 0.5*s2/dx**2; B = (r - q - 0.5*s2)/(2*dx)
    lo, up, di = A - B, A + B, -2*A - r
    ab = np.zeros((3, len(x)))
    ab[0, 1:] = -0.5*dt*up[:-1]; ab[1, :] = 1 - 0.5*dt*di; ab[2, :-1] = -0.5*dt*lo[1:]
    rhs = V.copy()
    rhs[1:-1] += 0.5*dt*(lo[1:-1]*V[:-2] + di[1:-1]*V[1:-1] + up[1:-1]*V[2:])
    ab[1, 0], ab[0, 1] = 1.0, 0.0; ab[1, -1], ab[2, -2] = 1.0, 0.0
    rhs[0] = V[0]*np.exp(-r*dt); rhs[-1] = V[-1]*np.exp(-r*dt)
    return solve_banded((1, 1), ab, rhs)

def pde_european(payoff, T, r, q, vol_fn, S_min=5., S_max=500.,
                 n_space=1201, n_time=600):
    x = np.linspace(np.log(S_min), np.log(S_max), n_space); S = np.exp(x)
    dt = T/n_time; V = payoff(S).astype(float)
    P = np.empty((n_time+1, n_space)); P[n_time] = V
    for n in range(n_time-1, -1, -1):
        sig = np.asarray(vol_fn(S, n*dt), float)*np.ones(n_space)
        V = _cn_step(V, S, x, dt, r, q, sig); P[n] = V
    return S, np.linspace(0, T, n_time+1), P, np.gradient(P, x, axis=1)/S[None, :]

def phoenix_pde(T, r, q, vol_fn, S0, obs, Ba, Bc, coupon, Bp,
                S_min=5., S_max=500., n_space=1201, n_per=300, shift=0.0):
    """Phoenix (no memory) by PDE, with jump conditions on the observation dates.
    `shift` replaces each indicator with a call spread of half-width shift."""
    x = np.linspace(np.log(S_min), np.log(S_max), n_space); S = np.exp(x); perf = S/S0
    ind = (lambda p, lv: (p >= lv).astype(float)) if shift <= 0 else \
          (lambda p, lv: np.clip((p - (lv - shift))/(2*shift), 0., 1.))
    dates = [0.0] + list(obs)
    ip = ind(perf, Bp)
    V = coupon*ind(perf, Bc) + ip*1.0 + (1 - ip)*np.minimum(perf, 1.0)
    grids, tims = [], []
    for k in range(len(obs)-1, -1, -1):
        t0, t1 = dates[k], dates[k+1]; dt = (t1 - t0)/n_per
        loc = np.empty((n_per+1, n_space)); loc[n_per] = V
        for n in range(n_per-1, -1, -1):
            sig = np.asarray(vol_fn(S, t0 + n*dt), float)*np.ones(n_space)
            V = _cn_step(V, S, x, dt, r, q, sig); loc[n] = V
        grids.append(loc); tims.append(np.linspace(t0, t1, n_per+1))
        if k > 0:
            ia = ind(perf, Ba)
            V = ia*(1 + coupon) + (1 - ia)*(V + coupon*ind(perf, Bc))
    grids, tims = grids[::-1], tims[::-1]
    P = np.vstack([g[:-1] for g in grids] + [grids[-1][-1:]])
    tt = np.concatenate([t[:-1] for t in tims] + [tims[-1][-1:]])
    return S, tt, P, np.gradient(P, x, axis=1)/S[None, :]

print("Toolbox loaded.")

### Tutorial parameters — your term sheet

The "market" is represented by a reference Heston model. It is a teaching device: it gives us
a realistic implied volatility surface **and** a known dynamics, which will later let us play
the role of reality in the hedging backtest.

**Set `GROUP` and `TEAM` below**, then run the whole notebook. The barriers of your term sheet
follow from the team table (also in the answer sheet, tab *Teams*). With `GROUP = None` you get
the reference case of the subject ($B_c = 70\,\%$, $B_p = 60\,\%$), whose numbers are the ones
quoted in the subject.

In [ ]:
GROUP = None   # "IF1", "IF2" or "IF3"; None = reference case (Bc = 70%, Bp = 60%)
TEAM  = None   # your team number, 1 to 8

# Team table: TEAMS[group][team - 1] = (Bc, Bp), coupon barrier and protection barrier
TEAMS = {
    "IF1": [(0.65, 0.55), (0.70, 0.65), (0.75, 0.50), (0.65, 0.65), (0.70, 0.50), (0.75, 0.55), (0.65, 0.50), (0.75, 0.65)],
    "IF2": [(0.65, 0.65), (0.70, 0.50), (0.75, 0.55), (0.65, 0.50), (0.70, 0.55), (0.75, 0.65), (0.65, 0.55), (0.70, 0.65)],
    "IF3": [(0.65, 0.50), (0.70, 0.55), (0.75, 0.65), (0.65, 0.55), (0.70, 0.65), (0.75, 0.50), (0.70, 0.50), (0.75, 0.55)],
}

S0, r, q = 100.0, 0.02, 0.02
HP = dict(v0=0.04, kappa=1.5, theta=0.04, sigma=0.6, rho=-0.7)   # reference model

T    = 3.0
OBS  = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
Ba   = 1.00                            # autocall barrier, the same for every team

if GROUP is None:
    Bc, Bp = 0.70, 0.60                # reference case of the subject
    TERM_SHEET = "reference case (no group, no team)"
else:
    _g = GROUP.strip().upper() if isinstance(GROUP, str) else GROUP
    if _g not in TEAMS:
        raise ValueError(f"GROUP = {GROUP!r} is not valid: write GROUP = \"IF1\", \"IF2\" or "
                         f"\"IF3\" (with the quotes), or GROUP = None for the reference case.")
    if isinstance(TEAM, bool) or not isinstance(TEAM, (int, np.integer)) \
            or not 1 <= TEAM <= len(TEAMS[_g]):
        raise ValueError(f"TEAM = {TEAM!r} is not valid: write your team number, an integer "
                         f"from 1 to {len(TEAMS[_g])} (e.g. TEAM = 3), without quotes.")
    Bc, Bp = TEAMS[_g][TEAM - 1]
    TERM_SHEET = f"group {_g}, team {TEAM}"

print(f"Term sheet in use: {TERM_SHEET}")
print(f"  autocall barrier   Ba = {Ba:.0%}")
print(f"  coupon barrier     Bc = {Bc:.0%}")
print(f"  protection barrier Bp = {Bp:.0%}")
if GROUP is None and TEAM is not None:
    print("  (TEAM is ignored because GROUP is None: set GROUP as well to use your term sheet.)")

ATM = implied_vol(heston_call([S0], T, S0, r, q, HP)[0], S0, S0, T, r, q)
print(f"\n3-year ATM implied volatility: {ATM:.2%}")
print(f"3-year forward               : {S0*np.exp((r-q)*T):.2f}")

---
## §1 — The reference market: the implied volatility surface

**→ Question Q10** (● team)

We extract the implied volatility surface from the reference model. It is *the only market
information* that all the models will have.

In [ ]:
Ks = np.array([50, 60, 70, 80, 90, 100, 110, 120, 130.])
mats = [0.5, 1.0, 2.0, 3.0]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for Tm in mats:
    iv = [implied_vol(p, S0, K, Tm, r, q)
          for K, p in zip(Ks, heston_call(Ks, Tm, S0, r, q, HP))]
    ax[0].plot(Ks/S0*100, np.array(iv)*100, 'o-', label=f'{Tm} y')
ax[0].set(xlabel='Strike (% of spot)', ylabel='Implied vol (%)',
          title='Implied volatility skew of the reference market')
ax[0].axvline(100, color='k', lw=0.8, ls=':'); ax[0].legend()

for Tm in mats:
    iv = [implied_vol(p, S0, K, Tm, r, q)
          for K, p in zip(Ks, heston_call(Ks, Tm, S0, r, q, HP))]
    ax[1].plot(Ks/S0*100, np.array(iv)*100 - ATM*100, 'o-', label=f'{Tm} y')
ax[1].axhline(0, color='k', lw=0.8); ax[1].axvline(Bp*100, color='r', ls='--',
          lw=1, label=f'barrier {Bp:.0%}')
ax[1].set(xlabel='Strike (% of spot)', ylabel='Gap to the 3-year ATM vol (pts)',
          title='What flat Black-Scholes ignores'); ax[1].legend()
plt.tight_layout(); plt.show()

iv_Bp = implied_vol(heston_call([Bp*S0], T, S0, r, q, HP)[0], S0, Bp*S0, T, r, q)
print(f"3-year implied vol at strike {Bp:.0%}: {iv_Bp:.2%}   (ATM: {ATM:.2%})")
print(f"--> a gap of {(iv_Bp-ATM)*100:.1f} volatility points at the protection barrier.")

---
## §2 — Dupire local volatility and Gyöngy's theorem

**→ Question Q10** (● team)

We build the local volatility surface with Dupire's formula, then check that a Monte Carlo
simulation under local volatility **reproduces the vanilla prices of the reference model**.

> ⏱ About 30 seconds.

In [ ]:
Ts_grid = np.linspace(0.05, 3.3, 40)
ys_grid = np.linspace(-1.1, 0.7, 61)
LV, W = build_local_vol(S0, r, q, HP, Ts_grid, ys_grid)

fig, ax = plt.subplots(figsize=(10, 4))
for Tm in [0.5, 1.0, 2.0, 3.0]:
    i = np.argmin(abs(Ts_grid - Tm)); F = S0*np.exp((r - q)*Tm)
    kk = np.linspace(40, 150, 120)
    ax.plot(kk, [np.sqrt(np.interp(np.log(k/F), ys_grid, LV.locvar[i]))*100 for k in kk],
            label=f'T = {Tm}')
ax.set(xlabel='Spot', ylabel='Local volatility (%)',
       title='Dupire local volatility surface'); ax.legend(); plt.show()

In [ ]:
# --- Validation: does local vol reproduce the market's vanillas?
NPATH, NSTEP = 200_000, 750
OBS_IDX = np.round(OBS/T*NSTEP).astype(int)          # indices of the observation dates

sim_lv = simulate_localvol(S0, r, q, LV, T, NSTEP, NPATH, seed=7, obs_idx=OBS_IDX)
ST = sim_lv[0][-1]                                    # spot at maturity

print(f"Local vol Monte Carlo: {NPATH:,} paths, {NSTEP} time steps (one per trading day)\n")
print(f"{'Strike':>7} {'Market (ref.)':>14} {'Local vol MC':>14} {'diff':>9}"
      f" {'IV ref.':>9} {'IV loc.':>9}")
for K in [60, 70, 80, 90, 100, 110, 120]:
    ref = heston_call([K], T, S0, r, q, HP)[0]
    mc  = np.exp(-r*T)*np.maximum(ST - K, 0).mean()
    print(f"{K:7.0f} {ref:14.4f} {mc:14.4f} {mc-ref:+9.4f}"
          f" {implied_vol(ref,S0,K,T,r,q):9.4f} {implied_vol(mc,S0,K,T,r,q):9.4f}")

> **Q10 (● team) — write this in your answer sheet.** How close are the local volatility
> prices to the market's, given the Monte Carlo precision? Which result explains it, and what
> exactly does it guarantee? Does it say anything about the joint law of
> $(S_{T_1}, S_{T_2})$, hence about a payoff that depends on more than one date?

---
## §3 — The European-barrier put: the models agree

**→ Question Q11** (● team)

The protection put of the Phoenix, with a barrier observed **at maturity only**, has a payoff
that depends on $S_T$ alone:

$$ \text{payoff} = \left(1 - \tfrac{S_T}{S_0}\right)\mathbf{1}_{\{S_T < B_p\,S_0\}} $$

M2 and M1 are priced by Monte Carlo (printed with their standard error); M3 is priced
semi-analytically (COS method), so it carries no Monte Carlo error.

In [ ]:
dip = lambda S: np.where(S/S0 < Bp, np.maximum(1 - S/S0, 0.0), 0.0)

sim_h  = simulate_heston(S0, r, q, HP, T, NSTEP, NPATH, seed=11, obs_idx=OBS_IDX)
sim_bs = simulate_bs(S0, r, q, ATM, T, NSTEP, NPATH, seed=11, obs_idx=OBS_IDX)

dip_prices = {
    'M3 stochastic vol (market)': heston_european(dip, T, S0, r, q, HP),
    'M2 local vol (Dupire)'     : np.exp(-r*T)*dip(sim_lv[0][-1]).mean(),
    'M1 flat Black-Scholes'     : np.exp(-r*T)*dip(sim_bs[0][-1]).mean(),
}
dip_se = {                                   # Monte Carlo standard errors (None: semi-analytic)
    'M3 stochastic vol (market)': None,
    'M2 local vol (Dupire)'     : np.exp(-r*T)*mc_se(dip(sim_lv[0][-1])),
    'M1 flat Black-Scholes'     : np.exp(-r*T)*mc_se(dip(sim_bs[0][-1])),
}
ref = dip_prices['M3 stochastic vol (market)']
print(f"European-barrier put (K=100%, H={Bp:.0%}, T=3 y), in % of notional")
print(f"M2 and M1: Monte Carlo, {NPATH:,} paths ({NPATH//2:,} antithetic pairs), {NSTEP} time steps."
      f"\nM3: semi-analytic (COS method), no Monte Carlo error.\n")
print(f"  {'Model':28s} {'price':>8s} {'std error':>14s}   {'gap vs market':>13s}")
for k, v in dip_prices.items():
    se = dip_se[k]
    se_txt = 'semi-analytic' if se is None else f"{se:.4%}"
    gap_se = '' if se is None else f"  (SE {se/ref:.1%})"
    print(f"  {k:28s} {v:8.4%} {se_txt:>14s}   {v/ref-1:+13.1%}{gap_se}")

> **Q11 (● team) — write this in your answer sheet**, with the numbers of your term sheet, the
> number of paths and the standard error. Interpret what you have just measured.

---
## §4 — The full Phoenix: the models diverge

**→ Questions Q12 to Q15** (Q12, Q13, Q14 ● team; Q15 ◆ individual)

We now add the autocall clause, which introduces **path dependence**. Nothing else changes:
the three models still calibrate exactly the same vanillas.

The first table observes the protection barrier at maturity only (European barrier); the second
observes it **at each close** (every simulated trading day), for Q14.

Each break-even coupon is printed per year with its Monte Carlo standard error (SE), obtained
by the delta method: with $a_i$ the coupon annuity of path $i$ (coupon indicator × discount
factor, summed over the dates, per unit of coupon) and $r_i$ its redemption leg,
$c = (1-\bar r)/\bar a$ and $\mathrm{SE}(c) = \mathrm{SE}\big(\overline{r + c\,a}\big)/\bar a$,
computed on antithetic pairs. M2 and M3 use independent random numbers, so the SE of their gap is
$\sqrt{\mathrm{SE}_2^2 + \mathrm{SE}_3^2}$.

In [ ]:
MODELS = {'M3 stochastic vol': sim_h, 'M2 local vol': sim_lv,
          'M1 flat Black-Scholes': sim_bs}

def coupon_table(american):
    # returns (break-even semi-annual coupons, their standard errors), one entry per model
    print(f"{'Model':24s} {'coupon/half-yr':>14s} {'coupon p.a.':>12s} {'SE p.a.':>9s}"
          f" {'annuity':>9s} {'redemption':>11s}")
    res, se = {}, {}
    for name, p in MODELS.items():
        _, ann, red = phoenix_mc(p, S0, r, OBS, Ba, Bc, 1.0, Bp, american=american)
        c = break_even_coupon(ann, red); res[name] = c
        se[name] = break_even_coupon_se(*phoenix_paths(p, S0, r, OBS, Ba, Bc, Bp,
                                                       american=american))[1]
        print(f"{name:24s} {c:14.3%} {2*c:12.3%} {2*se[name]*1e4:6.1f} bp"
              f" {ann:9.4f} {red:11.4f}")
    return res, se

print(f"Monte Carlo: {NPATH:,} paths per model ({NPATH//2:,} antithetic pairs), {NSTEP} time steps."
      f"\nSE p.a. = standard error of the annual coupon (delta method), in bp.\n")
print("=== EUROPEAN protection barrier (observed at maturity) ===")
res_eu, se_eu = coupon_table(False)
print("\n=== DAILY-CLOSE protection barrier (observed at each close) ===")
res_daily, se_daily = coupon_table(True)

for lab, res, se in [("European", res_eu, se_eu), ("daily-close", res_daily, se_daily)]:
    gap    = (res['M2 local vol'] - res['M3 stochastic vol'])*2*1e4
    se_gap = np.hypot(se['M2 local vol'], se['M3 stochastic vol'])*2*1e4
    print(f"\n{lab:11s} barrier: local vol / stochastic vol gap = {gap:+.0f} bp of annual coupon"
          f"  (SE {se_gap:.1f} bp)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
names = list(MODELS); x = np.arange(len(names)); wdt = 0.36
ax.bar(x - wdt/2, [res_eu[n]*200 for n in names], wdt, label='European barrier')
ax.bar(x + wdt/2, [res_daily[n]*200 for n in names], wdt, label='daily-close barrier')
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("Break-even coupon (% p.a.)")
ax.set_title("Same product, same calibrated vanillas, three coupons")
for i, n in enumerate(names):
    ax.text(i - wdt/2, res_eu[n]*200 + .12, f"{res_eu[n]*200:.2f}", ha='center', fontsize=9)
    ax.text(i + wdt/2, res_daily[n]*200 + .12, f"{res_daily[n]*200:.2f}", ha='center', fontsize=9)
ax.legend(); plt.tight_layout(); plt.show()

> **Q12–Q14 (● team) — write these in your answer sheet**, with the numbers of your term sheet,
> the number of paths and the standard error. Interpret what you have just measured.
> **Q15 (◆ individual)** goes into your individual written assignment.

---
## §4bis — Measuring the forward skew

**→ Question Q16** (● team)

The previous section measures a gap between M2 and M3 without explaining it. Here we measure
the forward smile. A **forward smile** $T_1 \to T_2$ is the implied volatility of
*forward-starting* options, with payoff $(S_{T_2}/S_{T_1} - k)^+$: it is the smile the market
will "see" from $T_1$.

Q16 asks how this object relates to the Phoenix: after an observation date on which the
product is not called, what does the remaining product look like, and do today's vanillas
constrain its value?

No additional simulation is needed: we reuse the paths of §4 on the observation dates
$t = 1$ year and $t = 3$ years.

In [ ]:
i1, i2 = 1, 5                       # OBS[1] = 1 year, OBS[5] = 3 years
tau_f = OBS[i2] - OBS[i1]           # forward horizon: 2 years
ks = np.array([0.7, 0.8, 0.9, 1.0, 1.1, 1.2])

smiles = {}
for name, sim in [('M3 stochastic vol', sim_h), ('M2 local vol', sim_lv)]:
    R = sim[0][i2]/sim[0][i1]       # forward return S_T2 / S_T1
    smiles[name] = np.array([
        implied_vol(np.exp(-r*tau_f)*np.maximum(R - k, 0).mean(), 1.0, k, tau_f, r, q)
        for k in ks])

# spot smile of the same length, for reference: both models calibrate it identically
Kspot = ks*S0
smiles['spot smile 0->2y (market)'] = np.array([
    implied_vol(p, S0, K, tau_f, r, q)
    for K, p in zip(Kspot, heston_call(Kspot, tau_f, S0, r, q, HP))])

slope = lambda v: (v[0] - v[4])*100          # vol(70%) - vol(110%), in vol points
print(f"{'k':>6}" + ''.join(f"{n[:25]:>27s}" for n in smiles))
for j, k in enumerate(ks):
    print(f"{k:6.1f}" + ''.join(f"{smiles[n][j]:27.2%}" for n in smiles))
print("\nSmile slope (vol 70% - vol 110%, in volatility points):")
for n, v in smiles.items():
    print(f"  {n:34s} {slope(v):5.2f}")
print(f"\n--> local vol flattens the forward skew by "
      f"{(1 - slope(smiles['M2 local vol'])/slope(smiles['M3 stochastic vol'])):.0%}.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for n, v in smiles.items():
    st = ':' if 'spot' in n else '-'
    ax[0].plot(ks*100, v*100, st, marker='o', lw=1.9, label=n)
ax[0].set(xlabel='Forward moneyness $k$ (%)', ylabel='Implied vol (%)',
          title='Forward smile 1 year $\\to$ 3 years')
ax[0].legend(fontsize=8)

ax[1].bar(range(3), [slope(v) for v in smiles.values()],
          color=['C0', 'C1', 'C7'], width=.55)
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(['M3\nstochastic vol', 'M2\nlocal vol',
                                                   '2-year spot\n(market)'])
ax[1].set(ylabel='Slope (vol points)', title='Does the forward skew survive?')
for i, v in enumerate(smiles.values()):
    ax[1].text(i, slope(v) + .15, f'{slope(v):.2f}', ha='center')
plt.tight_layout(); plt.show()

> **Q16 (● team) — write this in your answer sheet**, with the numbers of your term sheet and
> their precision. Interpret what you have just measured.

---
## §4ter — M4: the local-stochastic model, the one actually used

**→ Questions Q17** (○ optional) **and Q18** (● team)

The previous parts set M2 against M3. In practice **no desk prices an autocall with either
one**: the market standard is the **local-stochastic volatility (LSV)** model, which combines
the two.

$$\frac{dS_t}{S_t} = (r-q)\,dt + L(S_t,t)\sqrt{v_t}\,dW^1_t,
\qquad dv_t = \kappa(\theta - v_t)\,dt + \boldsymbol{\eta}\,\xi\sqrt{v_t}\,dW^2_t$$

The **leverage function** $L$ is calibrated so that the model reproduces the vanillas exactly;
by Gyöngy's theorem, this requires

$$L^2(S,t)\;\mathbb{E}^{\mathbb{Q}}\!\left[v_t \mid S_t = S\right] = \sigma_{\mathrm{loc}}^2(S,t).$$

We compute it with the **particle method** (Guyon–Henry-Labordère): we simulate a cloud of
particles, estimate $\mathbb{E}[v_t\mid S_t]$ by binning, and deduce $L$ time step by time step.

The parameter $\eta \in [0,1]$ is the **mixing fraction**: $\eta = 0$ gives back local
volatility, $\eta = 1$ stochastic volatility.

M4 is provided calibrated: in Q18 you validate it before using it. Each break-even coupon is
printed with its standard error, computed as in §4. All values of $\eta$ use the same random
numbers (seed 21). The standard error treats antithetic pairs as independent, which neglects the
weak interaction between particles through the estimate of $\mathbb{E}[v_t\mid S_t]$.

> ⏱ About 1 minute.

In [ ]:
def simulate_lsv(S0, r, q, hp, eta, lv, T, n_steps, n_paths, seed=0,
                 n_bins=80, obs_idx=None):
    # LSV calibrated by the particle method: L^2(S,t) E[v_t|S_t] = sigma_loc^2(S,t).
    rng = np.random.default_rng(seed); dt = T/n_steps; m = n_paths//2
    ka, th, xi, rh = hp['kappa'], hp['theta'], hp['sigma'], hp['rho']
    S = np.full(n_paths, float(S0)); v = np.full(n_paths, float(hp['v0']))
    full = obs_idx is None
    if full:
        out = np.empty((n_steps+1, n_paths)); out[0] = S
    else:
        ov = np.empty((len(obs_idx), n_paths)); rm = np.empty_like(ov)
        k = 0; run = np.full(n_paths, np.inf)
    for i in range(n_steps):
        t = i*dt
        sloc = lv(S, t, S0*np.exp((r - q)*t))
        if i == 0:
            Ev = np.full(n_paths, v[0])            # the whole cloud is at S0
        else:
            x = np.log(S); lo, hi = np.percentile(x, [0.2, 99.8])
            edges = np.linspace(lo, hi, n_bins+1)
            idx = np.clip(np.digitize(x, edges) - 1, 0, n_bins-1)
            cnt = np.bincount(idx, minlength=n_bins)
            sm  = np.bincount(idx, weights=v, minlength=n_bins)
            mean = np.where(cnt > 0, sm/np.maximum(cnt, 1), np.nan)
            ctr = 0.5*(edges[:-1] + edges[1:]); good = np.isfinite(mean)
            mean = np.interp(ctr, ctr[good], mean[good])
            mean = np.convolve(mean, np.ones(5)/5, mode='same')     # smoothing
            mean[:2] = mean[2]; mean[-2:] = mean[-3]
            Ev = np.interp(x, ctr, mean)
        L = np.clip(sloc/np.sqrt(np.maximum(Ev, 1e-6)), 0.1, 5.0)
        vp = np.maximum(v, 0.0); sig = L*np.sqrt(vp)
        z1 = rng.standard_normal(m); z2 = rng.standard_normal(m)
        z1 = np.concatenate([z1, -z1]); z2 = np.concatenate([z2, -z2])
        dW1 = z1*np.sqrt(dt); dW2 = (rh*z1 + np.sqrt(1 - rh**2)*z2)*np.sqrt(dt)
        S = S*np.exp((r - q - 0.5*sig**2)*dt + sig*dW1)
        v = v + ka*(th - vp)*dt + eta*xi*np.sqrt(vp)*dW2
        if full: out[i+1] = S
        else:
            run = np.minimum(run, S)
            if k < len(obs_idx) and (i+1) == obs_idx[k]:
                ov[k] = S; rm[k] = run; run = np.full(n_paths, np.inf); k += 1
    return out if full else (ov, rm)

In [ ]:
NL, NPL = 600, 100_000                       # 600 steps: the 6 dates fall on the grid
OIL = np.round(OBS/T*NL).astype(int)
etas = [0.0, 0.25, 0.5, 0.75, 1.0]
ks   = np.array([0.7, 0.8, 0.9, 1.0, 1.1, 1.2])

lsv = {}
for e in etas:
    lsv[e] = simulate_lsv(S0, r, q, HP, e, LV, T, NL, NPL, seed=21, obs_idx=OIL)

# 1) does every value of eta reproduce the vanillas?
print(f"M4: {NPL:,} particles ({NPL//2:,} antithetic pairs), {NL} time steps, seed 21 for every eta.\n")
print("3-year implied vol: does the mixing change the calibration?")
print(f"{'K':>5} {'market':>9}" + "".join(f"{'eta='+str(e):>10}" for e in etas))
for K in [60, 80, 100, 120]:
    ivm = implied_vol(heston_call([K], T, S0, r, q, HP)[0], S0, K, T, r, q)
    row = "".join(
        f"{implied_vol(np.exp(-r*T)*np.maximum(lsv[e][0][-1]-K,0).mean(),S0,K,T,r,q):10.2%}"
        for e in etas)
    print(f"{K:5.0f} {ivm:9.2%}" + row)

In [ ]:
# 2) the break-even coupon and the forward skew as functions of eta
def phoenix_from_obs(ov):
    n = ov.shape[1]; alive = np.ones(n, bool); pvc = np.zeros(n); pvr = np.zeros(n)
    for k, d in enumerate(OBS):
        perf = ov[k]/S0; df = np.exp(-r*d)
        pvc += np.where(alive & (perf >= Bc), df, 0.0)
        if k < len(OBS) - 1:
            cl = alive & (perf >= Ba); pvr += np.where(cl, df, 0.0); alive &= ~cl
        else:
            pvr += np.where(alive, np.where(perf < Bp, np.minimum(perf, 1.0), 1.0)*df, 0.0)
    return break_even_coupon(pvc.mean(), pvr.mean())

cpn_eta, slope_eta, se_eta = [], [], []
for e in etas:
    ov = lsv[e][0]
    cpn_eta.append(phoenix_from_obs(ov)*200)
    se_eta.append(break_even_coupon_se(*phoenix_paths((ov, None), S0, r, OBS, Ba, Bc, Bp))[1]*2e4)
    R = ov[5]/ov[1]
    sm = np.array([implied_vol(np.exp(-r*2.0)*np.maximum(R - k, 0).mean(), 1.0, k, 2.0, r, q)
                   for k in ks])
    slope_eta.append((sm[0] - sm[4])*100)

print(f"{'eta':>6} {'coupon p.a.':>13} {'SE p.a.':>9} {'fwd smile slope':>17}")
for e, cp, se, pe in zip(etas, cpn_eta, se_eta, slope_eta):
    print(f"{e:6.2f} {cp:12.3f}% {se:6.1f} bp {pe:17.2f}")
print(f"\nRange of the coupon as eta spans [0, 1]: {(max(cpn_eta)-min(cpn_eta))*100:.0f} bp"
      f" of annual coupon (see Q18).")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(etas, cpn_eta, 'o-', lw=2, color='C3')
ax[0].axhline(res_eu['M2 local vol']*200, ls=':', color='C1', label='M2 local vol')
ax[0].axhline(res_eu['M3 stochastic vol']*200, ls=':', color='C0', label='M3 stochastic vol')
ax[0].fill_between(etas, min(cpn_eta), cpn_eta, alpha=.12, color='C3')
ax[0].set(xlabel=r'Mixing fraction $\eta$', ylabel="Break-even coupon (% p.a.)",
          title=r"Break-even coupon as a function of $\eta$")
ax[0].legend(fontsize=8)

ax[1].plot(etas, slope_eta, 's-', lw=2, color='C2')
ax[1].axhline(slope(smiles['spot smile 0->2y (market)']), ls=':', color='k',
              label='2-year spot smile slope')
ax[1].set(xlabel=r'Mixing fraction $\eta$', ylabel='Forward smile slope (pts)',
          title=r'Forward smile slope as a function of $\eta$')
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

> **Q18 (● team) — write this in your answer sheet**, with the numbers of your term sheet, the
> number of paths and the standard errors. Interpret what you have just measured. The
> operational definition of the model uncertainty adjustment goes into the Decision tab of the team answer sheet
> (Q36). Q17 (○ optional) is a derivation and needs no number.

---
## §5 — Sign of vega

**→ Questions Q19 to Q21** (◆ individual, mandatory)

The seller is short the digital strip and long the put. Do the two effects offset each other?
We shift the implied volatility surface by ±1 point and rebuild the local volatility. The
building blocks are priced by PDE: there is no Monte Carlo error here.

> ⏱ About 30 seconds.

In [ ]:
def building_blocks(bump, Bc_=Bc):
    lvb, _ = build_local_vol(S0, r, q, HP, Ts_grid, ys_grid, iv_bump=bump)
    vf = lambda s, t: lvb(s, t, S0*np.exp((r - q)*t))
    S, _, P1, _ = phoenix_pde(T, r, q, vf, S0, OBS, Ba, Bc_, 1.0, Bp)
    S, _, P0, _ = phoenix_pde(T, r, q, vf, S0, OBS, Ba, Bc_, 0.0, Bp)
    red = np.interp(S0, S, P0[0])
    return np.interp(S0, S, P1[0]) - red, red    # (annuity, redemption leg)

for Bc_ in [Bc, 0.95]:
    out = {b: building_blocks(b, Bc_) for b in (-0.01, 0.0, 0.01)}
    dA = (out[0.01][0] - out[-0.01][0])/2
    dR = (out[0.01][1] - out[-0.01][1])/2
    print(f"--- coupon barrier {Bc_:.0%} ---")
    print(f"  digital strip annuity : {out[0.0][0]:.4f}   vega = {dA:+.5f} /vol pt")
    print(f"  redemption leg        : {out[0.0][1]:.4f}   vega = {dR:+.5f} /vol pt")
    for b in (-0.01, 0.0, 0.01):
        a, rr = out[b]
        print(f"     break-even coupon at vol {b*100:+.0f} pt: {break_even_coupon(a, rr)*200:6.3f}% p.a.")
    print()

In [ ]:
# Vega of a European digital (flat Black-Scholes vol 17%) as a function of its barrier
levels = np.linspace(0.50, 1.45, 60); v0_ = 0.17
def digit(B, v):
    K = B*S0
    return np.exp(-r*T)*(1 - norm.cdf((np.log(K/S0) - (r - q - 0.5*v**2)*T)/(v*np.sqrt(T))))
vega = [(digit(B, v0_ + 0.005) - digit(B, v0_ - 0.005)) for B in levels]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(levels*100, vega, lw=2); ax.axhline(0, color='k', lw=1)
ax.axvline(100, color='k', ls=':', lw=1)
ax.axvline(Bc*100, color='r', ls='--', lw=1.2, label=f'coupon barrier of your term sheet ({Bc:.0%})')
ax.fill_between(levels*100, vega, 0, where=np.array(vega) < 0, alpha=.15, color='C3')
ax.fill_between(levels*100, vega, 0, where=np.array(vega) > 0, alpha=.15, color='C2')
ax.set(xlabel='Barrier (% of spot)', ylabel='Vega (per vol point)',
       title="Vega of a European digital as a function of its barrier")
ax.legend(); plt.tight_layout(); plt.show()

> **Q19–Q21 (◆ individual): use these numbers in your written answers.** Does the sign of the
> total vega depend on the coupon barrier? Why?

---
## §6 — Choosing a model by its greeks

**→ Questions Q22 to Q24** (● team)

We turn the question around. A model is not only a price generator: it is a generator of
**hedging strategies**.

Protocol: "reality" follows the reference model M3. We delta-hedge the European-barrier put,
rebalancing every week for 3 years, with the deltas of M1, M2 and M3 in turn. Each hedger
starts with **its own** price. The table also gives the standard error of the mean P&L.

> ⏱ About 30 seconds.

In [ ]:
# Deltas of M1 and M2 by PDE; deltas of M3 on an (S, v, tau) grid by the COS method
vf_lv = lambda s, t: LV(s, t, S0*np.exp((r - q)*t))
Sg, tg, P_lv, D_lv = pde_european(dip, T, r, q, vf_lv)
_,  _,  P_bs, D_bs = pde_european(dip, T, r, q, lambda s, t: ATM)

Sgh  = np.exp(np.linspace(np.log(15.), np.log(320.), 140))
vgh  = np.linspace(0.004, 0.30, 16)
taug = np.linspace(0.0, T, 31)
Vh = np.zeros((len(taug), len(vgh), len(Sgh)))
for i, tau in enumerate(taug):
    if tau <= 1e-8:
        Vh[i] = dip(Sgh)[None, :]; continue
    for j, v in enumerate(vgh):
        p = dict(HP); p['v0'] = v
        a, b = _cos_range(tau, r, q, v, p['theta'])
        xg = np.linspace(a, b, 2000)
        dens = heston_density(xg, tau, r, q, N=1024, **p)
        Vh[i, j] = np.exp(-r*tau)*_trapz(dip(Sgh[:, None]*np.exp(xg)[None, :])
                                         * dens[None, :], xg, axis=1)
Dh = np.gradient(Vh, Sgh, axis=2)

itp = {'M3 stochastic vol': RegularGridInterpolator((taug, vgh, Sgh), Dh,
                                bounds_error=False, fill_value=None),
       'M2 local vol'      : RegularGridInterpolator((tg, Sg), D_lv,
                                bounds_error=False, fill_value=None),
       'M1 flat Black-Scholes': RegularGridInterpolator((tg, Sg), D_bs,
                                bounds_error=False, fill_value=None)}
P0 = {'M3 stochastic vol': float(RegularGridInterpolator((taug, vgh, Sgh), Vh)([[T, HP['v0'], S0]])[0]),
      'M2 local vol'      : float(np.interp(S0, Sg, P_lv[0])),
      'M1 flat Black-Scholes': float(np.interp(S0, Sg, P_bs[0]))}
print("Initial prices:", {k: f"{v:.4%}" for k, v in P0.items()})

In [ ]:
NH, NPH = 156, 40_000                      # weekly rebalancing
pth, vpt = simulate_heston(S0, r, q, HP, T, NH, NPH, seed=99, return_var=True)  # full paths
th = np.linspace(0, T, NH + 1); dth = T/NH

res_hedge = {}
for name, f in itp.items():
    cash = np.full(NPH, P0[name]); pos = np.zeros(NPH)
    for i in range(NH):
        S, v, t = pth[i], vpt[i], th[i]
        d = f(np.stack([np.full_like(S, T - t), v, S], -1)) if '3' in name \
            else f(np.stack([np.full_like(S, t), S], -1))
        cash -= (d - pos)*S; pos = d
        cash *= np.exp(r*dth); cash += pos*S*(np.exp(q*dth) - 1)
    pl = cash + pos*pth[-1] - dip(pth[-1])
    res_hedge[name] = (P0[name], pl.mean(), pl.std(),
                       np.percentile(pl, 5), np.percentile(pl, 1), pl)

print(f"Backtest: {NPH:,} paths of the reference model M3, {NH} weekly rebalancing dates.\n")
print(f"{'Deltas from':24s} {'price 0':>8s} {'mean P&L':>10s} {'SE mean':>8s} {'std dev':>9s}"
      f" {'q 5%':>9s} {'q 1%':>9s}")
for k, (a, b, c_, d, e, pl) in res_hedge.items():
    print(f"{k:24s} {a:8.2%} {b:+10.2%} {mc_se(pl):8.2%} {c_:9.2%} {d:+9.2%} {e:+9.2%}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, (_, _, _, _, _, pl) in res_hedge.items():
    ax[0].hist(pl*100, bins=140, range=(-40, 20), histtype='step', lw=1.6, label=k)
ax[0].axvline(0, color='k', lw=1)
ax[0].set(xlabel='Terminal hedging P&L (% of notional)', ylabel='frequency',
          title='Distribution of the hedging P&L'); ax[0].legend(fontsize=8)

kk = np.linspace(30, 160, 300); i0 = 0
ax[1].plot(kk, np.interp(kk, Sg, D_lv[i0]), label='M2 local vol', lw=1.8)
ax[1].plot(kk, np.interp(kk, Sg, D_bs[i0]), label='M1 flat BS', lw=1.8)
ax[1].plot(kk, itp['M3 stochastic vol'](np.stack([np.full_like(kk, T),
           np.full_like(kk, HP['v0']), kk], -1)), label='M3 stochastic vol', lw=1.8)
ax[1].axvline(Bp*S0, color='r', ls='--', lw=1, label=f'barrier {Bp:.0%}')
ax[1].set(xlabel='Spot', ylabel='Delta', title='Delta of the barrier put at $t=0$')
ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

> **Q22 and Q24 (● team) — write these in your answer sheet**, with the numbers of your term
> sheet, the number of paths and their precision. Interpret what you have just measured.
> Q23 is discussed orally in class.

---
## §7 — Delta at the barriers and *barrier shift*

**→ Questions Q25 to Q27** (Q25, Q26 ● team; Q27 ○ optional)

Your team runs this section itself. The delta is in the unit of (R8): change in value for a
+100 % move of the spot, as a fraction of the notional ($S\,\partial V/\partial S$). Everything
here is computed by PDE (local volatility M2): there is no Monte Carlo error.

> ⏱ About 1 minute.

In [ ]:
CPN = res_eu['M2 local vol']      # local vol break-even coupon (§4, European barrier)
grid = np.arange(Bp*S0 - 12., Bp*S0 + 14., 1.)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for sh, style in [(0.0, '-'), (0.03, '--')]:
    S, tt2, P, D = phoenix_pde(T, r, q, vf_lv, S0, OBS, Ba, Bc, CPN, Bp,
                               shift=sh, n_per=800)
    for t_, col in zip([2.5, 2.9, 2.99, 2.999], ['C0', 'C1', 'C2', 'C3']):
        i = np.argmin(abs(tt2 - t_))
        ax[0 if sh == 0 else 1].plot(grid, np.interp(grid, S, D[i]*S), style,
                                     color=col, lw=1.8, label=f't = {t_}')
for a, ttl in zip(ax, ['No smoothing', 'With a 3% barrier shift']):
    a.axvline(Bp*S0, color='r', ls=':', lw=1.2)
    a.set(xlabel='Spot', ylabel='Delta (change for +100% of spot)', title=ttl)
    a.legend(fontsize=8)
ax[0].set_ylim(-1, None); ax[1].set_ylim(-1, None)
plt.tight_layout(); plt.show()

print(f"{'shift':>7s} {'|delta| max':>12s} {'break-even coupon p.a.':>24s}")
for sh in [0.0, 0.005, 0.01, 0.02, 0.03, 0.05]:
    S, tt2, P, D = phoenix_pde(T, r, q, vf_lv, S0, OBS, Ba, Bc, CPN, Bp,
                               shift=sh, n_per=800)
    m = (S > Bp*S0 - 12) & (S < Bp*S0 + 15)
    f = lambda cc: np.interp(S0, S, phoenix_pde(T, r, q, vf_lv, S0, OBS, Ba, Bc,
                                                cc, Bp, shift=sh)[2][0]) - 1.0
    print(f"{sh:7.1%} {np.abs(D[-2][m]*S[m]).max():12.1f} {brentq(f,0,.15,xtol=1e-7)*200:23.3f}%")

In [ ]:
# Delta near the autocall barrier, just before an observation date
S, tt2, P, D = phoenix_pde(T, r, q, vf_lv, S0, OBS, Ba, Bc, CPN, Bp, n_per=800)
gg = np.linspace(88, 115, 200)
fig, ax = plt.subplots(figsize=(9, 4))
for t_ in [0.0, 0.30, 0.45, 0.49]:
    i = np.argmin(abs(tt2 - t_))
    ax.plot(gg, np.interp(gg, S, D[i]*S), lw=1.8, label=f't = {t_}')
ax.axvline(Ba*S0, color='r', ls='--', lw=1.2, label="autocall barrier (100%)")
ax.axhline(0, color='k', lw=1)
ax.set(xlabel='Spot', ylabel='Delta', title="Delta near the autocall barrier before the first observation date")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

> **Q25–Q26 (● team) — write these in your answer sheet**, with the numbers of your term sheet
> and the grid used. Interpret what you have just measured. Q27 (○ optional) uses the last plot.

---
## §8 — Block D: a pricer written by an AI

**→ Questions D1 to D5** (● team): D1 (test plan) and D2 in class during TD 2; D3, D4 and D5
in the Decision tab of the team answer sheet.

Your team's module `phoenix_pricer_llm.py` was generated by a code assistant from the
description of *your* term sheet and of the reference market. It runs, it is documented and it
prints a break-even coupon. You are the validator.

**How you get the module.** Once your team has handed in its test plan (D1), the instructor
emails the module to your team's Block D lead, who shares it with the rest of the team. Save
`phoenix_pricer_llm.py` in the same folder as this notebook, and run the notebook with your
`GROUP` and `TEAM` so that both describe the same term sheet.

1. **Before receiving the module**, write your test plan in the Markdown cell below, with the
   time: at least five tests, one per error class (numerical scheme, market data, payoff logic,
   term-sheet reading, greeks), each with its expected result, justified without the module.
2. Code your tests in the next cell (D2). You can rely on the §0 engine (`heston_call`,
   `simulate_heston`, `phoenix_mc`), which serves as an independent reference. The module's
   interface is in the subject (Block D).
   **Warning:** the module calls the vol of vol `xi` (`llm.HESTON`), whereas the notebook calls
   it `sigma` (`HP`): do not pass the notebook's parameters to the module directly.
3. For each error found (D3): fix it in a copy of the file, then measure its impact in basis
   points of annual break-even coupon, **with the same random draws**.

> Block D is the only part where you may use an AI, and only to help write your tests (AI is not allowed elsewhere). You remain solely responsible for what you hand in:
> state in the Decision tab which tools were used and what you checked yourselves.

### Our test plan (written BEFORE running the module) — time: ……

| # | Test | Expected result | Error class targeted |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |

In [ ]:
import phoenix_pricer_llm as llm

# Starting point: what the module claims.
c_llm = llm.break_even_coupon(seed=42)
print(f"Break-even coupon claimed by the module: {2*c_llm:.3%} per year")

# --- Your tests below -----------------------------------------------------------
# Example structure:
# def test_martingale():
#     paths = llm.simulate_heston(100.0, 3.0, 300, 100_000, 0.02, 0.02, llm.HESTON, seed=1)
#     ...

---
## Going further (○ optional, not graded)

A few extensions, if you want to dig deeper after the sessions:

1. **Memory feature.** `phoenix_mc(sim, ..., memory=True)` turns on the catch-up of missed
   coupons. By how much does the break-even coupon fall? Does the gap between models change?
2. **Step-down autocall barrier.** Modify `phoenix_mc` so that `Ba` decreases by 1 % per
   period. What is the effect on the expected life and on the coupon?
3. **Sensitivity to vol of vol.** Vary `HP['sigma']` from 0.3 to 0.9, recalibrating the local
   volatility each time. Plot the M2/M3 coupon gap: it is the *direct measure* of model risk, as
   a function of the strength of the dynamics that local volatility cannot represent.
4. **Worst-of.** Extend to two correlated underlyings. Does the model risk on correlation add
   to the model risk on volatility, or compound it?
5. **Rebalancing.** Redo the §6 backtest with monthly, then daily rebalancing. What happens to
   the share of residual risk due to discretization, relative to the share due to the *gap*?